# Initial Setup

In [ ]:
! git clone https://github.com/huseyin-karaca/s2t-tr-dev
%cd /content/s2t-tr-dev

from google.colab import files
files.view('/content/s2t-tr-dev')

!make create_environment
!make requirements

from google.colab import userdata
import os

# Pull some env variables from secrets. It is only necessary
# for fast downloads and git push. Otherwise, not a problem.
os.environ["GITHUB_TOKEN"] = userdata.get('GitHubPAT')
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')


# Başarıyla giriş yapıp yapmadığını test et!
!gh auth status

# bunlar da gerekli sanırım:
!git config --global user.name "huseyin-karaca"
!git config --global user.email "huseyinkaraccca@gmail.com"

!gh auth setup-git



# when a commit needed:
#!git status
#!git add .
#!git commit -m "colab changes v2"
#!git push origin main

Cloning into 's2t-tr-dev'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 158 (delta 0), reused 2 (delta 0), pack-reused 153 (from 1)
Receiving objects: 100% (158/158), 220.91 MiB | 53.64 MiB/s, done.
Resolving deltas: 100% (52/52), done.
Updating files: 100% (61/61), done.
/content/s2t-tr-dev


<IPython.core.display.Javascript object>

uv venv --python 3.10
Using CPython 3.10.12 interpreter at: /usr/bin/python3.10
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate
>>> New uv virtual environment created. Activate with:
>>> Windows: .\.venv\Scripts\activate
>>> Unix/macOS: source ./.venv/bin/activate
uv sync
Resolved 199 packages in 12ms
Prepared 195 packages in 38.73s
Installed 195 packages in 312ms
 + absl-py==2.4.0
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + anyio==4.13.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + async-timeout==5.0.1
 + attrs==26.1.0
 + audioread==3.1.0
 + babel==2.18.0
 + beautifulsoup4==4.14.3
 + bleach==6.3.0
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.7
 + click==8.3.2
 + comm==0.2.3
 + contourpy==1.3.2
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.3
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + datasets==3.6.0
 + debugpy==1.8

In [ ]:
from google.colab import drive
drive.mount('/gdrive')

Mounted at /gdrive


In [ ]:
!cp /content/s2t-tr-dev/data/processed/edinburghcstr_ami/combined_features.parquet "/gdrive/My Drive/s2t-tr-dev/data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet"

In [ ]:
!mkdir -p data/processed/edinburghcstr_ami
!cp "/gdrive/My Drive/s2t-tr-dev/data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet" /content/s2t-tr-dev/data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet

In [ ]:
# 1. Baseline (mevcut): primary + hard CE karışımı
!uv run python -m src.training.train \
        --parquet-path data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet \
        --batch-size 128 \
        --max-epochs 50 \
        --num-workers 11 \
        --experiment-name ami_full_w2v2_large

2026-04-24 11:18:31.291 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
Seed set to 42
2026-04-24 11:18:37,326 [INFO] httpx: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"
2026-04-24 11:18:37,355 [INFO] src.data.dataset: Loaded 12612 samples from data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet
2026-04-24 11:18:37,356 [INFO] __main__: Dataset splits: train=10089, val=1261, test=1262
2026-04-24 11:18:37,399 [INFO] __main__: === Model Parameter Counts ===
2026-04-24 11:18:37,399 [INFO] __main__:                  projection:     656128
2026-04-24 11:18:37,399 [INFO] __main__:                pos_encoding:          0
2026-04-24 11:18:37,399 [INFO] __main__:            model_embeddings:        768
2026-04-24 11:18:37,399 [INFO] __main__:                  cls_tokens:        768
2026-04-24 11:18:37,399 [INFO] __main__:              stage1_encoder:    1054208
2026-

In [ ]:
!git add .
!git commit -m "w2v2 changed to large & robust, performance increased, now other loss functions will be tested"
!git push origin main

In [ ]:
%cd s2t-tr-dev

/content/s2t-tr-dev


train.py updated for the following runs:

In [ ]:
# 2. Sadece weighted WER
!uv run python -m src.training.train \
        --parquet-path data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet \
        --batch-size 16 \
        --max-epochs 50 \
        --num-workers 11 \
        --experiment-name ami_full_w2v2_large_loss_wer_only \
        --primary-weight 1.0 \
        --aux-ce-weight 0.0 \
        --soft-ce-weight 0.0


2026-04-24 13:30:03.207 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
Seed set to 42
2026-04-24 13:30:16,260 [INFO] httpx: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"
Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
!uv run python -m src.training.train \
    --parquet-path data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet \
    --batch-size 4 \
    --num-workers 0 \
    --experiment-name test_run

2026-04-24 13:28:44.407 | INFO     | src.config:<module>:9 - PROJ_ROOT path is: /content/s2t-tr-dev
Seed set to 42
2026-04-24 13:28:50,490 [INFO] httpx: HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/parquet/parquet.py "HTTP/1.1 404 Not Found"
Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
# 3. Sadece hard CE (classification)
!uv run python -m src.training.train \
        --parquet-path data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet \
        --batch-size 64 \
        --max-epochs 50 \
        --num-workers 11 \
        --experiment-name ami_full_w2v2_large_loss_ce_only \
        --primary-weight 0.0 \
        --aux-ce-weight 1.0 \
        --soft-ce-weight 0.0

In [ ]:
# 4. Sadece soft CE (WER-aware classification)
!uv run python -m src.training.train \
        --parquet-path data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet \
        --batch-size 64 \
        --max-epochs 50 \
        --num-workers 11 \
        --experiment-name ami_full_w2v2_large_loss_softce_T01 \
        --primary-weight 0.0 \
        --aux-ce-weight 0.0 \
        --soft-ce-weight 1.0 \
        --soft-ce-temperature 0.1

In [ ]:
# 5. Primary + soft CE (en umut verici teorik karışım)
!uv run python -m src.training.train \
        --parquet-path data/processed/edinburghcstr_ami/combined_features-w2v2-large.parquet \
        --batch-size 64 \
        --max-epochs 50 \
        --num-workers 11 \
        --experiment-name ami_full_w2v2_large_loss_wer_softce \
        --primary-weight 1.0 \
        --aux-ce-weight 0.0 \
        --soft-ce-weight 0.5 \
        --soft-ce-temperature 0.1